# STG_5 — Ingeniería de features del diputado

Construye el dataset de entrenamiento (`df_entrenamiento.csv`) a partir de:
- `data/df_modelado.csv` — 28.738 votos consolidados
- `data/df_features_titulo.csv` — embeddings semánticos de los 1.022 títulos únicos

**Regla de oro**: toda feature de historial se calcula **solo con votos anteriores** al voto que se predice (sin fuga de información).


In [ ]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

# Rutas
DATA_DIR = Path("../data")


In [ ]:
# T1 — Carga de datos
df_votos = pd.read_csv(DATA_DIR / "df_modelado.csv", parse_dates=["fecha_votacion"])
df_titulos = pd.read_csv(DATA_DIR / "df_features_titulo.csv")

print(f"df_modelado   → {len(df_votos):,} filas | {df_votos['diputado'].nunique()} diputados únicos")
print(f"df_titulos    → {len(df_titulos):,} filas | {df_titulos['titulo_base'].nunique()} títulos únicos")
print()
print("Columnas df_modelado:", list(df_votos.columns))
print()
print("Rango de fechas:", df_votos['fecha_votacion'].min().date(), "→", df_votos['fecha_votacion'].max().date())
print()
print("Distribución de votos:")
print(df_votos['voto'].value_counts())


In [ ]:
# Verificaciones T1
assert len(df_votos) == 28738, f"Se esperaban 28.738 filas, hay {len(df_votos)}"
assert df_titulos['titulo_base'].nunique() == 1022, f"Se esperaban 1.022 títulos únicos"
assert df_votos['fecha_votacion'].isna().sum() == 0, "Hay NaN en fecha_votacion"

print("✓ T1 PASA: 28.738 filas y 1.022 títulos verificados.")


## Limpieza previa: filtrar AUSENTE y normalizar ABSTENCIÓN

In [ ]:
# Los votos AUSENTE no representan posición política → se filtran
# ABSTENCION y ABSTENCIÓN son el mismo voto → se unifican
antes = len(df_votos)
df_votos = df_votos[df_votos['voto'] != 'AUSENTE'].copy()
df_votos['voto'] = df_votos['voto'].replace('ABSTENCION', 'ABSTENCIÓN')

print(f"Filas eliminadas (AUSENTE): {antes - len(df_votos):,}")
print(f"Filas restantes           : {len(df_votos):,}")
print()
print("Distribucion final de votos:")
print(df_votos['voto'].value_counts().to_string())


## T2 — Merge de embeddings

In [ ]:
# T2 — Unir tema_id, tema_label y embeddings desde df_features_titulo
# NOTA: df_modelado.csv NO contiene tema_id; viene solo de df_features_titulo.csv
emb_cols = [c for c in df_titulos.columns if c.startswith('emb_')]
print(f"Columnas de embeddings: {len(emb_cols)}")

df = df_votos.merge(
    df_titulos[['titulo_base', 'tema_id', 'tema_label'] + emb_cols],
    on='titulo_base',
    how='left'
)

print(f"Filas tras merge: {len(df):,}")
print(f"NaN en embeddings: {df[emb_cols].isna().sum().sum()}")
print(f"NaN en tema_id   : {df['tema_id'].isna().sum()}")

assert len(df) == len(df_votos), "El merge cambio la cantidad de filas"
assert df[emb_cols].isna().sum().sum() == 0, "Hay NaN en los embeddings tras el merge"
assert df['tema_id'].isna().sum() == 0, "Hay NaN en tema_id tras el merge"

print("T2 PASA: merge correcto, 0 NaN en embeddings y tema_id.")


## T3 — `es_afirmativo` y `alineado_bloque`

In [ ]:
# es_afirmativo: 1 si el diputado votó AFIRMATIVO, 0 si no
df['es_afirmativo'] = (df['voto'] == 'AFIRMATIVO').astype(float)

# voto_bloque: para cada (titulo, fecha, bloque), el voto más frecuente del bloque
voto_bloque = (
    df.groupby(['titulo_base', 'fecha_votacion', 'bloque'])['voto']
    .agg(lambda x: x.mode().iloc[0])
    .reset_index()
    .rename(columns={'voto': 'voto_bloque'})
)
df = df.merge(voto_bloque, on=['titulo_base', 'fecha_votacion', 'bloque'], how='left')

# alineado_bloque: 1 si el diputado votó igual que la moda de su bloque
df['alineado_bloque'] = (df['voto'] == df['voto_bloque']).astype(float)

print(f"es_afirmativo  — media: {df['es_afirmativo'].mean():.3f} | NaN: {df['es_afirmativo'].isna().sum()}")
print(f"alineado_bloque — media: {df['alineado_bloque'].mean():.3f} | NaN: {df['alineado_bloque'].isna().sum()}")

assert df['es_afirmativo'].isna().sum() == 0
assert df['alineado_bloque'].isna().sum() == 0
assert len(df) == 25082

print("T3 PASA.")


## T4 — Tasas históricas del diputado (sin leakage)

In [ ]:
def media_acumulada_pasado(serie):
    """
    Para cada posicion i, devuelve la media de los valores 0..i-1.
    Posicion 0 devuelve NaN (no hay historia previa).
    Nunca incluye el valor actual -> cero leakage.
    """
    cumsum   = serie.cumsum().shift(1)
    cumcount = serie.notna().cumsum().shift(1)
    return cumsum / cumcount

# Ordenar por diputado y fecha (obligatorio para que shift opere en orden temporal)
df = df.sort_values(['diputado', 'fecha_votacion']).reset_index(drop=True)

# tasa_afirmativo_hist: % de votos AFIRMATIVO del diputado en toda su historia previa
df['tasa_afirmativo_hist'] = (
    df.groupby('diputado')['es_afirmativo']
    .transform(media_acumulada_pasado)
)

# tasa_afirmativo_tema_hist: idem pero solo para el tema_id especifico
df['tasa_afirmativo_tema_hist'] = (
    df.groupby(['diputado', 'tema_id'])['es_afirmativo']
    .transform(media_acumulada_pasado)
)

# tasa_alineacion_bloque_hist: % de veces que el diputado voto igual que su bloque
df['tasa_alineacion_bloque_hist'] = (
    df.groupby('diputado')['alineado_bloque']
    .transform(media_acumulada_pasado)
)

for col in ['tasa_afirmativo_hist', 'tasa_afirmativo_tema_hist', 'tasa_alineacion_bloque_hist']:
    nan_count = df[col].isna().sum()
    media = df[col].mean()
    print(f"{col}: media={media:.3f} | NaN={nan_count:,}")


## T5 — Tasas de comportamiento con ventana temporal (sin leakage)

In [ ]:
# Fechas de corte definidas en la spec
CORTE_2023 = pd.Timestamp('2023-12-10')  # inicio del gobierno actual
CORTE_2026 = pd.Timestamp('2026-01-01')  # inicio de 2026

# Estrategia vectorizada: para cada fila, multiplicar es_afirmativo por una mascara
# (1 si la fecha esta en la ventana, 0 si no), luego acumular y desplazar 1 posicion.
# Asi cada fila ve el acumulado de las filas anteriores que caen en la ventana.

for col_out, corte in [
    ('tasa_afirmativo_desde_2023', CORTE_2023),
    ('tasa_afirmativo_2026',       CORTE_2026),
]:
    col_mask  = f'_mask_{col_out}'
    col_afirm = f'_afirm_{col_out}'

    df[col_mask]  = (df['fecha_votacion'] >= corte).astype(float)
    df[col_afirm] = df['es_afirmativo'] * df[col_mask]

    cumsum_afirm = df.groupby('diputado')[col_afirm].transform(lambda x: x.cumsum().shift(1))
    cumsum_mask  = df.groupby('diputado')[col_mask].transform(lambda x: x.cumsum().shift(1))

    df[col_out] = cumsum_afirm / cumsum_mask  # 0/0 = NaN cuando no hay historia en la ventana

    df.drop(columns=[col_mask, col_afirm], inplace=True)

for col in ['tasa_afirmativo_desde_2023', 'tasa_afirmativo_2026']:
    non_nan = df[col].notna().sum()
    print(f"{col}: media={df[col].mean():.3f} | NaN={df[col].isna().sum():,} | no-NaN={non_nan:,}")

print()
# Verificar no-leakage: para cada diputado, el primer voto desde cada corte debe ser NaN
for col, corte in [('tasa_afirmativo_desde_2023', CORTE_2023), ('tasa_afirmativo_2026', CORTE_2026)]:
    primeros_en_ventana = (
        df[df['fecha_votacion'] >= corte]
        .groupby('diputado')
        .head(1)
    )
    assert primeros_en_ventana[col].isna().all(), f"LEAKAGE en {col}"

print("T5 PASA: primer voto de cada diputado en cada ventana es NaN.")


## T6 — `n_votos_hist`: cantidad de votos previos del diputado

In [ ]:
# cumcount() empieza en 0 para el primer voto de cada diputado,
# lo que representa exactamente "cuantos votos anteriores tiene".
df['n_votos_hist'] = df.groupby('diputado').cumcount()

print(f"n_votos_hist — min={df['n_votos_hist'].min()} | max={df['n_votos_hist'].max()} | NaN={df['n_votos_hist'].isna().sum()}")
assert df['n_votos_hist'].isna().sum() == 0
assert df['n_votos_hist'].min() == 0  # primer voto de cada diputado = 0
print("T6 PASA.")


## T7 — `tasa_afirmativo_bloque_tema`: historial del bloque por tema

In [ ]:
# Riesgo resuelto: si hicieramos shift(1) fila por fila dentro del grupo (bloque, tema_id),
# el 'anterior' podria ser otro diputado votando EL MISMO DIA (no el dia anterior).
# Solucion: agregar primero por dia, hacer el shift a nivel de dia, y pegar de vuelta.

# Paso 1: suma y conteo por (bloque, tema_id, fecha)
daily = (
    df.groupby(['bloque', 'tema_id', 'fecha_votacion'])['es_afirmativo']
    .agg(suma='sum', n='count')
    .reset_index()
    .sort_values(['bloque', 'tema_id', 'fecha_votacion'])
)

# Paso 2: acumulado corrido un dia hacia atras (shift a nivel de fecha, no de fila individual)
daily['suma_acum'] = daily.groupby(['bloque', 'tema_id'])['suma'].transform(
    lambda x: x.cumsum().shift(1)
)
daily['n_acum'] = daily.groupby(['bloque', 'tema_id'])['n'].transform(
    lambda x: x.cumsum().shift(1)
)
daily['tasa_afirmativo_bloque_tema'] = daily['suma_acum'] / daily['n_acum']

# Paso 3: pegar de vuelta al dataset principal
df = df.merge(
    daily[['bloque', 'tema_id', 'fecha_votacion', 'tasa_afirmativo_bloque_tema']],
    on=['bloque', 'tema_id', 'fecha_votacion'],
    how='left'
)

nan_count = df['tasa_afirmativo_bloque_tema'].isna().sum()
print(f"tasa_afirmativo_bloque_tema: media={df['tasa_afirmativo_bloque_tema'].mean():.3f} | NaN={nan_count:,}")
assert nan_count > 0, "Deberia haber NaN para los primeros votos de cada bloque-tema"
print("T7 PASA.")


## T8 — Chequeo automático de no-leakage (ANTES del relleno)

In [ ]:
# Features que deben ser NaN en el primer voto de cada diputado.
# Si alguna no lo es, hay leakage y la notebook frena aqui.
FEATURES_HIST = [
    'tasa_afirmativo_hist',
    'tasa_alineacion_bloque_hist',
    'tasa_afirmativo_desde_2023',
    'tasa_afirmativo_2026',
]

primeros_votos = df.groupby('diputado').head(1)
errores = []

for col in FEATURES_HIST:
    n_no_nan = primeros_votos[col].notna().sum()
    if n_no_nan > 0:
        errores.append(f"  LEAKAGE en '{col}': {n_no_nan} diputados tienen valor en su primer voto")

# tasa_afirmativo_tema_hist: el primer voto de cada (diputado, tema_id) debe ser NaN
primeros_tema = df.groupby(['diputado', 'tema_id']).head(1)
n_no_nan_tema = primeros_tema['tasa_afirmativo_tema_hist'].notna().sum()
if n_no_nan_tema > 0:
    errores.append(f"  LEAKAGE en 'tasa_afirmativo_tema_hist': {n_no_nan_tema} combinaciones diputado-tema tienen valor en su primer voto")

# tasa_afirmativo_bloque_tema: el primer voto de cada (bloque, tema_id, fecha) debe ser NaN
primeros_bloque_tema = df.groupby(['bloque', 'tema_id', 'fecha_votacion']).head(1)
# (no aplica: bloque_tema NaN es esperado para el primer dia de cada bloque-tema, no para cada fila)

if errores:
    raise AssertionError("CHEQUEO DE NO-LEAKAGE FALLIDO:
" + "
".join(errores))

print("=" * 60)
print("CHEQUEO DE NO-LEAKAGE: TODO PASA")
print("=" * 60)
for col in FEATURES_HIST:
    n_nan = primeros_votos[col].isna().sum()
    print(f"  {col}: {n_nan}/{len(primeros_votos)} primeros votos son NaN (correcto)")
print(f"  tasa_afirmativo_tema_hist: {primeros_tema['tasa_afirmativo_tema_hist'].isna().sum()}/{len(primeros_tema)} primeros votos por tema son NaN (correcto)")
print("=" * 60)


## T9 — Cascada de relleno de NaN (honesta, sin leakage)

In [ ]:
# Regla: cada feature cae a la mas general disponible.
# En ultimo recurso: 0.5 neutro (NO la media global del dataset, que miraria el futuro).
# n_votos_hist ya tiene 0 NaN; no necesita relleno.

ANTES = {col: df[col].isna().sum() for col in [
    'tasa_afirmativo_hist', 'tasa_afirmativo_tema_hist',
    'tasa_alineacion_bloque_hist', 'tasa_afirmativo_desde_2023',
    'tasa_afirmativo_2026', 'tasa_afirmativo_bloque_tema'
]}

# 1. tema → hist individual (cuando no hay historia en ese tema, usar la tasa global del diputado)
df['tasa_afirmativo_tema_hist'] = df['tasa_afirmativo_tema_hist'].fillna(df['tasa_afirmativo_hist'])

# 2. ventanas → hist individual (cuando no hay votos en la ventana, usar la tasa global)
df['tasa_afirmativo_desde_2023'] = df['tasa_afirmativo_desde_2023'].fillna(df['tasa_afirmativo_hist'])
df['tasa_afirmativo_2026']       = df['tasa_afirmativo_2026'].fillna(df['tasa_afirmativo_hist'])

# 3. bloque-tema → hist individual (cuando el bloque no tiene historia en ese tema)
df['tasa_afirmativo_bloque_tema'] = df['tasa_afirmativo_bloque_tema'].fillna(df['tasa_afirmativo_hist'])

# 4. Residual → 0.5 neutro (primerísimos votos donde no hay ninguna historia previa)
FEATURES_MODELADO = [
    'tasa_afirmativo_hist', 'tasa_afirmativo_tema_hist',
    'tasa_alineacion_bloque_hist', 'tasa_afirmativo_desde_2023',
    'tasa_afirmativo_2026', 'tasa_afirmativo_bloque_tema'
]
for col in FEATURES_MODELADO:
    df[col] = df[col].fillna(0.5)

print("NaN antes y despues del relleno:")
print(f"{'Feature':<35} {'Antes':>7} {'Despues':>8}")
print("-" * 52)
for col in FEATURES_MODELADO:
    despues = df[col].isna().sum()
    print(f"{col:<35} {ANTES[col]:>7,} {despues:>8,}")

assert all(df[col].isna().sum() == 0 for col in FEATURES_MODELADO), "NaN residuales tras el relleno"
print()
print("T9 PASA: 0 NaN residuales en todas las features.")


## T10 — Encoding de `bloque` y `provincia`

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
import joblib

enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
df[['bloque_enc', 'provincia_enc']] = enc.fit_transform(df[['bloque', 'provincia']])

joblib.dump(enc, DATA_DIR / 'encoder_bloque_provincia.joblib')

print(f"Bloques distintos  : {df['bloque'].nunique()} -> codigos {int(df['bloque_enc'].min())} a {int(df['bloque_enc'].max())}")
print(f"Provincias distintas: {df['provincia'].nunique()} -> codigos {int(df['provincia_enc'].min())} a {int(df['provincia_enc'].max())}")
print(f"NaN en bloque_enc  : {df['bloque_enc'].isna().sum()}")
print(f"NaN en provincia_enc: {df['provincia_enc'].isna().sum()}")
print()
print("Encoder guardado en data/encoder_bloque_provincia.joblib")

assert df['bloque_enc'].isna().sum() == 0
assert df['provincia_enc'].isna().sum() == 0
print("T10 PASA.")


## T11 — Guardar `df_entrenamiento.csv` y verificación final

In [ ]:
emb_cols = [c for c in df.columns if c.startswith('emb_')]

FEATURES = (
    ['tema_id'] +
    emb_cols +
    ['bloque_enc', 'provincia_enc'] +
    ['tasa_afirmativo_hist', 'tasa_afirmativo_tema_hist', 'tasa_alineacion_bloque_hist',
     'tasa_afirmativo_desde_2023', 'tasa_afirmativo_2026', 'n_votos_hist',
     'tasa_afirmativo_bloque_tema']
)
TARGET = 'voto'
META   = ['diputado', 'titulo_base', 'fecha_votacion', 'bloque', 'provincia', 'tema_label']

cols_guardar = META + FEATURES + [TARGET]
df_out = df[cols_guardar].copy()

df_out.to_csv(DATA_DIR / 'df_entrenamiento.csv', index=False)

print("=" * 60)
print("VERIFICACION FINAL — df_entrenamiento.csv")
print("=" * 60)
print(f"Filas         : {len(df_out):,}")
print(f"Columnas      : {len(df_out.columns)} ({len(META)} meta + {len(FEATURES)} features + 1 target)")
print(f"Features      : {len(FEATURES)} ({1} tema_id + {len(emb_cols)} embeddings + 2 encoding + 7 historicas)")
print(f"NaN en features: {df_out[FEATURES].isna().sum().sum()}")
print(f"Votos distintos: {df_out[TARGET].unique()}")
print(f"Distribucion de votos:")
print(df_out[TARGET].value_counts().to_string())
print("=" * 60)

assert len(df_out) == 25082, f"Se esperaban 25.082 filas, hay {len(df_out)}"
assert df_out[FEATURES].isna().sum().sum() == 0, "Hay NaN en las features"
print("T11 PASA: dataset de entrenamiento guardado correctamente.")
